# PRBCD miss-condition experiments — self-contained

This notebook generates every dataset- and victim-model object required by the RQ1 experiments before starting PRBCD. It uses the same victim-training pipeline and cache format as the uploaded master-thesis notebook.

Phases:
1. Configure the dataset and victim models.
2. Train or load one victim model per seed and build `SEED_CONTEXTS`.
3. Run paired small/large discovery attacks.
4. Probe large-only and random-unseen candidates in small-run states.
5. Inject useful missed candidates and compare against random controls.
6. Optionally run the retention-policy ablation.

The diagnostics implementation is loaded as a local in-memory overlay. Repository source files are not modified.


## Dataset and victim-model generation

These cells reproduce the data-generation path used in the uploaded master-thesis notebook. They create the victim-model cache expected by `experiment_global_attack_direct` and construct all in-memory seed contexts used for validation and graph metadata.


In [ ]:
from networkx.convert import to_networkx_graph
from torch_geometric.utils import to_networkx


import os
from pathlib import Path

%load_ext autoreload
%autoreload 2

PROJECT_DIR = Path.cwd().parent
os.chdir(PROJECT_DIR)

print("PROJECT_DIR:", PROJECT_DIR)
print("Current directory:", Path.cwd())

import numpy as np


from matplotlib import pyplot as plt

from experiments import (
    experiment_train,
    experiment_local_attack_direct,
    experiment_global_attack_direct
)
import helpers.selector_pipeline_helpers

from sparse_smoothing.utils import load_and_standardize
from sparse_smoothing.models import GCN

files = ["cache/demo.json", "cache/demo/demo_1.pt", "cache/evasion_global_adj.json", "cache/evasion_global_attr.json", "cache/evasion_global_adj/evasion_global_adj_1.pt", "cache/evasion_global_attr/evasion_global_attr_1.pt"]

for file_path in files:
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"{file_path} has been deleted.")
    else:
        print(f"{file_path} does not exist.")

%matplotlib inline




In [ ]:
# ============================================================
# Dataset and victim-model configuration
# ============================================================

DATASET = "cora_ml"
SEEDS = [0,1,2,3,4]

if not SEEDS:
    raise ValueError("SEEDS must contain at least one integer seed.")
SEEDS = [int(seed) for seed in SEEDS]
SEED = SEEDS[0]
N_SEEDS = len(SEEDS)

MODEL_NAME = "GCN"
MODEL_LABEL = "GCN"
MODEL_STORAGE_TYPE = "demo_custom_split"

DROPOUT_VICTIM = 0.5
LR_VICTIM = 1e-2
WEIGHT_DECAY_VICTIM = 1e-3
PATIENCE_VICTIM = 300
MAX_EPOCHS_VICTIM = 3000

# experiment_train uses the repository cache. Existing compatible
# victim models may be reused by the underlying storage layer.
VICTIM_DEVICE = "cpu"
VICTIM_DATA_DEVICE = "cpu"

DATA_DIR = PROJECT_ROOT / "data"
CACHE_DIR = PROJECT_ROOT / "cache"
DATASET_PATH = DATA_DIR / f"{DATASET}.npz"

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset file not found: {DATASET_PATH}. "
        "Place the dataset in the repository data folder."
    )

CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Configured dataset={DATASET}, seeds={SEEDS}, model={MODEL_LABEL}")


In [ ]:
def set_global_seed(seed: int, deterministic: bool = True) -> None:
    """Seed Python, NumPy and PyTorch before every independent run."""
    seed = int(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def activate_seed_context(context: dict) -> None:
    """Expose one trained victim context through legacy notebook globals."""
    keys = [
        "seed", "train_statistics", "clean_acc", "model", "graph",
        "idx_train", "idx_val", "idx_test", "device", "n_nodes",
        "n_edges_directed", "n_undirected", "attr_matrix",
        "adj_matrix", "labels_raw", "edge_index", "edge_weight",
        "attr", "labels",
    ]
    for key in keys:
        if key in context:
            globals()["SEED" if key == "seed" else key] = context[key]


def aggregate_over_seeds(
    frame: pd.DataFrame,
    group_cols: list[str],
    metric_cols: list[str] | None = None,
) -> pd.DataFrame:
    """Return mean/std/SEM and number of distinct victim seeds."""
    if frame is None or frame.empty:
        return pd.DataFrame()
    if "seed" not in frame.columns:
        raise KeyError("Every raw table must contain a seed column.")
    missing = [column for column in group_cols if column not in frame.columns]
    if missing:
        raise KeyError(f"Missing grouping columns: {missing}")
    if metric_cols is None:
        metric_cols = [
            column for column in frame.select_dtypes(include=[np.number]).columns
            if column not in set(group_cols) | {"seed"}
        ]
    metric_cols = [column for column in metric_cols if column in frame.columns]
    if not group_cols:
        row = {"n_seeds": int(frame["seed"].nunique())}
        for metric in metric_cols:
            values = pd.to_numeric(frame[metric], errors="coerce")
            count = int(values.notna().sum())
            std = float(values.std(ddof=1)) if count > 1 else 0.0
            row[f"{metric}_mean"] = float(values.mean()) if count else np.nan
            row[f"{metric}_std"] = std
            row[f"{metric}_count"] = count
            row[f"{metric}_sem"] = std / np.sqrt(max(1, count))
        return pd.DataFrame([row])
    grouped = frame.groupby(group_cols, dropna=False)
    stats = grouped[metric_cols].agg(["mean", "std", "count"])
    stats.columns = [f"{metric}_{stat}" for metric, stat in stats.columns]
    stats = stats.reset_index()
    for metric in metric_cols:
        stats[f"{metric}_std"] = stats[f"{metric}_std"].fillna(0.0)
        stats[f"{metric}_sem"] = stats[f"{metric}_std"] / np.sqrt(
            stats[f"{metric}_count"].clip(lower=1)
        )
    seed_counts = grouped["seed"].nunique().rename("n_seeds").reset_index()
    return stats.merge(seed_counts, on=group_cols, how="left")


def add_mean_std_band(ax, x, mean, std, *, label=None, **plot_kwargs):
    x = np.asarray(x)
    mean = np.asarray(mean, dtype=float)
    std = np.nan_to_num(np.asarray(std, dtype=float), nan=0.0)
    line = ax.plot(x, mean, label=label, **plot_kwargs)[0]
    ax.fill_between(x, mean - std, mean + std, alpha=0.18, color=line.get_color())
    return line

set_global_seed(SEED)


In [ ]:
# ============================================================
# Train/load victim models and build all RQ1 seed contexts
# ============================================================

seed_contexts = []
train_curve_rows = []
train_summary_rows = []

for seed in SEEDS:
    set_global_seed(seed)
    print(f"Training or loading victim model for seed={seed}")

    stats = experiment_train.run(
        data_dir=str(DATA_DIR),
        dataset=DATASET,
        model_params=dict(
            label=MODEL_LABEL,
            model=MODEL_NAME,
            do_cache_adj_prep=True,
            n_filters=64,
            dropout=DROPOUT_VICTIM,
            svd_params=None,
            jaccard_params=None,
            gdc_params={"alpha": 0.15, "k": 64},
        ),
        train_params=dict(
            lr=LR_VICTIM,
            weight_decay=WEIGHT_DECAY_VICTIM,
            patience=PATIENCE_VICTIM,
            max_epochs=MAX_EPOCHS_VICTIM,
        ),
        binary_attr=False,
        make_undirected=True,
        seed=int(seed),
        artifact_dir=str(CACHE_DIR),
        model_storage_type=MODEL_STORAGE_TYPE,
        ppr_cache_params=dict(),
        device=VICTIM_DEVICE,
        data_device=VICTIM_DATA_DEVICE,
        display_steps=100,
        debug_level="info",
        custom_split_ratios=None,
    )

    model_seed = stats["model"]
    graph_seed = stats["graph"]
    idx_train_seed = stats["idx_train"]
    idx_val_seed = stats["idx_val"]
    idx_test_seed = stats["idx_test"]
    model_seed.eval()

    attr_matrix_seed, adj_matrix_seed, labels_raw_seed = graph_seed[:3]
    model_device = next(model_seed.parameters()).device
    row, col, value = adj_matrix_seed.coo()
    edge_index_seed = torch.stack([row, col], dim=0).long().to(model_device)
    edge_weight_seed = (
        torch.ones(edge_index_seed.size(1), dtype=torch.float32, device=model_device)
        if value is None else value.float().to(model_device)
    )
    attr_seed = attr_matrix_seed.float().to(model_device)
    labels_seed = labels_raw_seed.long().to(model_device)
    n_nodes_seed = int(adj_matrix_seed.sizes()[0])
    n_edges_directed_seed = int(adj_matrix_seed.nnz())
    n_undirected_seed = n_edges_directed_seed // 2
    clean_acc_seed = float(stats["accuracy"])

    context = {
        "seed": int(seed),
        "train_statistics": stats,
        "clean_acc": clean_acc_seed,
        "model": model_seed,
        "graph": graph_seed,
        "idx_train": idx_train_seed,
        "idx_val": idx_val_seed,
        "idx_test": idx_test_seed,
        "device": model_device,
        "n_nodes": n_nodes_seed,
        "n_edges_directed": n_edges_directed_seed,
        "n_undirected": n_undirected_seed,
        "attr_matrix": attr_matrix_seed,
        "adj_matrix": adj_matrix_seed,
        "labels_raw": labels_raw_seed,
        "edge_index": edge_index_seed,
        "edge_weight": edge_weight_seed,
        "attr": attr_seed,
        "labels": labels_seed,
    }
    seed_contexts.append(context)

    for split, values in (("train", stats.get("trace_train", [])),
                          ("validation", stats.get("trace_val", []))):
        for epoch, loss in enumerate(values, start=1):
            train_curve_rows.append({
                "seed": int(seed), "split": split,
                "epoch": int(epoch), "loss": float(loss),
            })

    train_summary_rows.append({
        "seed": int(seed),
        "clean_accuracy": clean_acc_seed,
        "n_train_epochs": len(stats.get("trace_train", [])),
        "n_val_epochs": len(stats.get("trace_val", [])),
    })

SEED_CONTEXTS = seed_contexts
SEED_CONTEXT_BY_SEED = {context["seed"]: context for context in seed_contexts}
activate_seed_context(SEED_CONTEXTS[0])

if len(SEED_CONTEXTS) != len(SEEDS):
    raise RuntimeError("Not every configured seed produced a victim context.")

shape_signatures = {
    (context["n_nodes"], context["n_undirected"])
    for context in SEED_CONTEXTS
}
if len(shape_signatures) != 1:
    raise RuntimeError(
        "The graph structure differs across victim seeds; paired edge IDs "
        "would not be comparable."
    )

train_curve_df = pd.DataFrame(train_curve_rows)
train_summary_raw_df = pd.DataFrame(train_summary_rows)
train_summary_df = aggregate_over_seeds(
    train_summary_raw_df,
    group_cols=[],
    metric_cols=["clean_accuracy", "n_train_epochs", "n_val_epochs"],
)

display(train_summary_df)
print("Generated SEED_CONTEXTS for:", sorted(SEED_CONTEXT_BY_SEED))
print("Victim cache:", CACHE_DIR)
print("Graph nodes:", SEED_CONTEXTS[0]["n_nodes"])
print("Undirected edges:", SEED_CONTEXTS[0]["n_undirected"])


In [ ]:
MISS_EPSILON = 0.01
# Increase this only when every configured block remains larger than the resulting attack budget.

SMALL_BLOCK_SIZES = [250,500,1_000,2000,3000,4000]
LARGE_BLOCK_SIZE = 50_000

EPOCHS = 300
FINE_TUNE_EPOCHS = 250
N_RESAMPLING_EPOCHS = EPOCHS - FINE_TUNE_EPOCHS
WITH_EARLY_STOPPING = False

REPEATS_PER_SEED = 1

# Checkpoints are evaluated after the update/projection of the corresponding
# epoch and before resampling.
CHECKPOINT_EPOCHS = sorted({
    epoch
    for epoch in [0, 9, 24, N_RESAMPLING_EPOCHS - 1, 99, EPOCHS - 1]
    if 0 <= epoch < EPOCHS
})

# Candidate construction for the counterfactual probe replay.
N_TOP_SIGNAL_LARGE_ONLY = 100
N_RANDOM_LARGE_ONLY = 100
N_RANDOM_UNSEEN_CONTROLS = 100
MAX_LARGE_FINAL_MISSED = 100
MAX_TOTAL_PROBES = 400

# A positive delta means that the one-edge-replaced block has a larger
# post-step PRBCD attack loss than the unchanged block after the same step.
POSITIVE_DELTA_TOL = 1e-7
ROBUST_POSITIVE_PROBABILITY = 0.60
ROBUST_MEDIAN_DELTA = 1e-5

# Causal injection experiment.
N_INJECTION_EDGES = 10

# Retention details can be large. The notebook samples at most this many
# candidates from each resampling event for plotting.
STORE_RESAMPLE_EDGE_DETAILS = True
RETENTION_ROWS_PER_EVENT = 5_000

# Expensive phases can be switched independently.
RUN_DISCOVERY_PHASE = True
RUN_PROBE_PHASE = True
RUN_INJECTION_PHASE = True
RUN_RETENTION_ABLATION = False

RETENTION_POLICIES = [
    "native",
    "random_keep",
    "reverse_keep",
    "full_resample",
]
RETENTION_ABLATION_BLOCK_SIZES = [
    min(SMALL_BLOCK_SIZES),
    LARGE_BLOCK_SIZE,
]

ARTIFACT_DIR = str(PROJECT_ROOT / "cache")
PERT_ADJ_STORAGE_TYPE = "evasion_global_adj"
PERT_ATTR_STORAGE_TYPE = "evasion_global_attr"

BASE_OUT_DIR = (
    PACKAGE_DIR
    / "outputs"
    / "prbcd_miss_condition_experiments"
)
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = BASE_OUT_DIR / f"{DATASET}__{RUN_ID}"
RAW_DIR = OUT_DIR / "raw_diagnostics"
PLOT_DIR = OUT_DIR / "plots"

for directory in [OUT_DIR, RAW_DIR, PLOT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

(BASE_OUT_DIR / "latest_run.txt").write_text(
    str(OUT_DIR.resolve()),
    encoding="utf-8",
)

## Phase 1 — paired discovery runs

These runs establish accuracy, realized coverage, resampling turnover, and
large-only candidate sets. They do not label any edge as harmful.

### Discovery plots

## Phase 2 — candidate construction and counterfactual probe replays

For each paired small/large run, candidate groups are built without calling
them harmful:

- `large_final_missed`: selected by the large run and never seen by the small.
- `large_top_signal`: large-only edges with the largest learned weights and
  positive-gradient accumulation.
- `large_only_random`: random edges seen only by the large run.
- `random_unseen_control`: uniform edges seen by neither paired run.

The small run is then replayed with the same seed. At fixed checkpoints each
candidate receives exact discrete and relaxed marginal-loss measurements plus
its gradient at epsilon.

### Aggregate contextual harmfulness

### Probe plots

## Phase 3 — causal injection

For each small run, select the strongest robustly harmful large-only edges.
Inject them at the checkpoint where their average measured effect was largest.
Compare against the same number of random-unseen control edges injected at the
same epoch. The baseline is the original small discovery run.

### Injection plots